# Feature Engineering and Analysis

### Import Libraries

In [1]:
import fiona
import geopandas as gpd
import numpy as np
import os
import pandas as pd
from pathlib import Path

## 1. Load the Transshipment Datasets

In [2]:
# CWD is the notebooks directory, as that is where Jupyter will run its kernel in.
TRANSSHIPMENT_DIR_FROM_CWD = "../data/raw/transshipment"
ENCOUNTER_EVENTS_FILENAME = "encounter-events-v20170717.csv"
LOITERING_EVENTS_FILENAME = "loitering-events-v20170717.csv"
VESSELS_FILENAME = "transshipment-vessels-v20170717.csv"

In [3]:
transshipment_data_dir = os.path.join(os.getcwd(), TRANSSHIPMENT_DIR_FROM_CWD)

encounter_path = os.path.join(transshipment_data_dir, ENCOUNTER_EVENTS_FILENAME)
loitering_path = os.path.join(transshipment_data_dir, LOITERING_EVENTS_FILENAME)
transshipment_path = os.path.join(transshipment_data_dir, VESSELS_FILENAME)

encounter_df = pd.read_csv(encounter_path)
loitering_df = pd.read_csv(loitering_path)
transshipment_df = pd.read_csv(transshipment_path)


print("="*20 + " Encounter " + "="*20)
print("Shape: ",encounter_df.shape)
print("Example Row: ",encounter_df.head(1))
print("="*20 + "=========" + "="*20)

print()

print("="*20 + " Loitering " + "="*20)
print("Shape: ",loitering_df.shape)
print("Example Row: ",loitering_df.head(1))
print("="*20 + "=========" + "="*20)

print()

print("="*20 + " Transshipment " + "="*20)
print("Shape: ",transshipment_df.shape)
print("Example Row: ",transshipment_df.head(1))
print("="*20 + "=========" + "="*20)

==================== Encounter ====================
Shape:  (11681, 9)
Example Row:     fishing_vessel_mmsi  transshipment_vessel_mmsi            start_time  \
0            416565000                  354240000  2016-11-18T14:30:00Z   

               end_time  mean_latitude  mean_longitude  duration_hr  \
0  2016-11-19T01:50:00Z     -17.039085      -79.063725    11.333333   

   median_distance_km  median_speed_knots  
0            0.038188            0.585402  

==================== Loitering ====================
Shape:  (46569, 9)
Example Row:     transshipment_mmsi  starting_latitude  starting_longitude  ending_latitude  \
0           256064000            12.7938          -69.503235         12.72795   

   ending_longitude    starting_timestamp      ending_timestamp  \
0         -69.73912  2017-09-06T02:11:32Z  2017-09-06T15:44:15Z   

   median_speed_knots  total_event_duration  
0            1.091065             13.915556  

==================== Transshipment ====================


## 2. Load the Global Marine Protected Area Data

In [4]:
MPA_GDB_PATH_FROM_CWD = "../data/raw/global_mpa/WDPA_WDOECM_Apr2026_Public_marine.gdb"

In [5]:
mpa_gdb_path = os.path.join(os.getcwd(), MPA_GDB_PATH_FROM_CWD)

fiona.listlayers(mpa_gdb_path)

['WDPA_WDOECM_poly_Apr2026_marine',
 'WDPA_WDOECM_point_Apr2026_marine',
 'WDPA_WDOECM_source_Apr2026_marine']

In [6]:
mpa_point_gdf = gpd.read_file(mpa_gdb_path, layer="WDPA_WDOECM_point_Apr2026_marine")

In [7]:
mpa_location = mpa_point_gdf[['NAME_ENG', 'geometry']]

# Extract latitude and longitude
mpa_location['geometry'] = mpa_location['geometry'].explode(ignore_index=True)
mpa_location['lat'] = mpa_location['geometry'].y.astype(float)
mpa_location['long'] = mpa_location['geometry'].x.astype(float)
mpa_location.head()

,NAME_ENG,geometry,lat,long
0,Byers Peninsula,POINT (-61.05 -62.617),-62.6170,-61.0500
1,Vallée du Fango,POINT (8.77 42.57),42.5700,8.7700
2,Malindi-Watamu,POINT (40.1944 -3.3411),-3.3411,40.1944
3,Aleutian Islands,POINT (-167.89 53.89),53.8900,-167.8900
4,The Virginia Coast Reserve,POINT (-75.78 37.33),37.3300,-75.7800


## 3. Engineer Features
### Loitering Events

In [8]:
loitering_new = loitering_df.copy()

#### 1. Add Distance Traveled

In [9]:
loitering_new['diff_lat'] = loitering_new['ending_latitude'] - loitering_new['starting_latitude']
loitering_new['diff_long'] = loitering_new['ending_longitude'] - loitering_new['starting_longitude']
loitering_new['movement_mag'] = np.sqrt(
    loitering_new['diff_lat']**2 + loitering_new['diff_long']**2
)

#### 2. Add Whether Event was at Night

In [10]:
# Determine if activity occured during night by correcting time (Local solar hour = UTC + longitude/15) % 24
loitering_new['starting_timestamp'] = pd.to_datetime(loitering_new['starting_timestamp'])
loitering_new['local_hour'] = (
    loitering_new['starting_timestamp'].dt.hour +
    loitering_new['starting_longitude'] / 15
) % 24

# Redefine is_night (True if between 6 PM and 6 AM)
loitering_new['is_night_corrected'] = (
    (loitering_new['local_hour'] >= 18) |
    (loitering_new['local_hour'] < 6)
).astype(int)

#### 3. Add Distance to Nearest Marine Protected Area

In [11]:
loitering_new['nearest_mpa_name'] = ""
loitering_new['nearest_mpa_lat'] = 0
loitering_new['nearest_mpa_long'] = 0
loitering_new['nearest_mpa_dist'] = 0

loit_coords = loitering_new[['starting_latitude', 'starting_longitude']].values
mpa_coords = mpa_location[['lat', 'long']].values

indices = [np.linalg.norm(mpa_coords - point, axis=1).argmin() for point in loit_coords]

nearest_mpas = mpa_location.iloc[indices]
loitering_new['nearest_mpa_name'] = nearest_mpas['NAME_ENG'].values
loitering_new['nearest_mpa_lat'] = nearest_mpas['lat'].values
loitering_new['nearest_mpa_long'] = nearest_mpas['long'].values

loitering_new['nearest_mpa_dist'] = np.sqrt(
    (loitering_new['starting_latitude'] - loitering_new['nearest_mpa_lat'])**2 +
    (loitering_new['starting_longitude'] - loitering_new['nearest_mpa_long'])**2
)

In [15]:
loitering_new.info()

<class 'pandas.DataFrame'>
RangeIndex: 46569 entries, 0 to 46568
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype              
---  ------                --------------  -----              
 0   transshipment_mmsi    46569 non-null  int64              
 1   starting_latitude     46569 non-null  float64            
 2   starting_longitude    46569 non-null  float64            
 3   ending_latitude       46569 non-null  float64            
 4   ending_longitude      46569 non-null  float64            
 5   starting_timestamp    46569 non-null  datetime64[us, UTC]
 6   ending_timestamp      46569 non-null  str                
 7   median_speed_knots    46569 non-null  float64            
 8   total_event_duration  46569 non-null  float64            
 9   diff_lat              46569 non-null  float64            
 10  diff_long             46569 non-null  float64            
 11  movement_mag          46569 non-null  float64            
 12  local_hour     

## 4. Save Engineered Datasets

In [13]:
PROCESSED_TRSHMNT_DIR_FROM_CWD = "../data/processed/transshipment"
PROCESSED_LOITERING_FILENAME = "loitering-events.csv"

cwd_path = Path(os.getcwd())
processed_trshmnt_path = cwd_path / PROCESSED_TRSHMNT_DIR_FROM_CWD

processed_trshmnt_path.mkdir(parents=True, exist_ok=True)

In [14]:
new_loitering_path = processed_trshmnt_path / PROCESSED_LOITERING_FILENAME
loitering_new.to_csv(new_loitering_path, index=False)